### 1. Conectar con Google Drive e Importar Librerías
Ejecuta la siguiente celda para dar acceso a Colab a tus archivos de Drive e importar `pandas` y `numpy`.

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

### 2. Configurar la Ruta y Listar Archivos
Por defecto, al montar Google Drive, tu unidad principal se encuentra en `/content/drive/My Drive/`.

IMPORTANTE => Hacer un acceso directo de la carpeta Datos a tu unidad para que el programa pueda leer los archivos.

Estructura de carpetas dentro de `Datos`:
- `Datos/Crudo`: datos originales (no se usan en este notebook).
- `Datos/Procesado`: datos limpios y consolidados que exportó el notebook de Data Understanding (sección 25). **Esta es la carpeta de entrada de este notebook.**

In [ ]:
RUTA_DATOS = '/content/drive/My Drive/Datos/Procesado'

if os.path.isdir(RUTA_DATOS):
    print(f"¡Éxito! Carpeta detectada en: {RUTA_DATOS}\n")
    archivos = sorted(f for f in os.listdir(RUTA_DATOS) if f.endswith(('.parquet', '.csv')))
    print(f"Se encontraron {len(archivos)} archivos de datos procesados:")
    for i, archivo in enumerate(archivos, 1):
        peso = os.path.getsize(os.path.join(RUTA_DATOS, archivo)) / 1024**2
        print(f"{i}. {archivo} ({peso:.1f} MB)")
else:
    print("Aún no se detectó la carpeta. Por favor verifica que exista 'Mi Unidad/Datos/Procesado' "
          "(se crea al ejecutar la sección 25 del notebook de Data Understanding) y vuelve a ejecutar esta celda.")

### 3. Lectura de los datos procesados
Se leen los archivos exportados en la sección 25 de Data Understanding. Cada archivo se carga en el DataFrame que usa el resto del notebook:

| Archivo | DataFrame | Contenido |
|---|---|---|
| `consolidado_marzo_junio_2025` | `df_consolidado` | Archivos de marzo-junio 2025 unidos y ordenados por `hora_inicio` |
| `concentrado_consumido` | `concentradoConsumido` | Consumo de concentrado por vaca |
| `produccion_leche_sesion` | `prodLecheSesion` | Producción por sesión (histórico) con tipos convertidos, sin filtrar |
| `s0_base_limpia` | `s0` | Ordeños filtrados + variables temporales y de lactancia |
| `s1_ventanas_robot` | `s1` | Robot × ventana de 30 min |
| `s2_sesiones_etapa` | `s2` | Ordeños con días en leche conocidos |
| `s3_episodios_cola` | `s3` | Episodios de cola |
| `s4_vaca_dia` | `s4` | Vaca × día |

Funciona con archivos `.parquet` o `.csv` (según el `FORMATO_EXPORTACION` que se haya usado al exportar).

In [ ]:
ARCHIVOS = {
    # nombre del archivo (sin extensión): nombre del DataFrame
    "consolidado_marzo_junio_2025": "df_consolidado",
    "concentrado_consumido": "concentradoConsumido",
    "produccion_leche_sesion": "prodLecheSesion",
    "s0_base_limpia": "s0",
    "s1_ventanas_robot": "s1",
    "s2_sesiones_etapa": "s2",
    "s3_episodios_cola": "s3",
    "s4_vaca_dia": "s4",
}

def leer_procesado(nombre):
    """Lee un archivo de Datos/Procesado (parquet o csv). Devuelve None si no existe."""
    for ext in ("parquet", "csv"):
        ruta = os.path.join(RUTA_DATOS, f"{nombre}.{ext}")
        if os.path.exists(ruta):
            return pd.read_parquet(ruta) if ext == "parquet" else pd.read_csv(ruta, low_memory=False)
    return None

print("Iniciando la lectura de archivos...")
datos = {}
for nombre, variable in ARCHIVOS.items():
    d = leer_procesado(nombre)
    if d is None:
        print(f"No se encontró '{nombre}' en {RUTA_DATOS}")
        continue
    datos[variable] = d
    print(f"✔ {variable:<22} <- {nombre}: {len(d):,} filas x {d.shape[1]} columnas")

print(f"\nArchivos cargados correctamente: {len(datos)} de {len(ARCHIVOS)}")

### 4. Restaurar tipos de datos
Al guardar en Drive algunos tipos cambian (`mes` se exporta como texto `YYYY-MM`, y las columnas con tipos mezclados se guardan como texto; con CSV todas las fechas vuelven como texto). Aquí se restauran:
- `mes` -> periodo mensual.
- `fecha` -> fecha (`datetime64`, a las 00:00 h).
- `hora_inicio`, `fin`, `inicio` y `ventana` -> fecha y hora.

In [ ]:
COLS_FECHA_HORA = ["hora_inicio", "fin", "inicio", "ventana"]

def restaurar_tipos(d):
    d = d.copy()
    if "mes" in d.columns:
        d["mes"] = pd.to_datetime(d["mes"], errors="coerce").dt.to_period("M")
    if "fecha" in d.columns:
        d["fecha"] = pd.to_datetime(d["fecha"], errors="coerce")
    for c in COLS_FECHA_HORA:
        if c in d.columns:
            d[c] = pd.to_datetime(d[c], errors="coerce")
    return d

datos = {variable: restaurar_tipos(d) for variable, d in datos.items()}

# Dejar cada DataFrame como variable del notebook
df_consolidado = datos.get("df_consolidado")
concentradoConsumido = datos.get("concentradoConsumido")
prodLecheSesion = datos.get("prodLecheSesion")
s0, s1, s2, s3, s4 = (datos.get(k) for k in ["s0", "s1", "s2", "s3", "s4"])

print("Tipos restaurados.")

### 5. Parámetros del notebook
Todos los umbrales y funciones auxiliares que se usan en más de una sección se definen aquí una sola vez. Si se cambia un valor, se cambia en todo el notebook.

In [ ]:
# ==== Parámetros del notebook ====
UMBRAL_COLA_MIN = 2                    # ≤ 2 min libres entre ordeños se considera "posible cola"
VENTANA = "30min"                      # tamaño del bloque de tiempo por robot
MIN_VENTANA = pd.Timedelta(VENTANA).total_seconds() / 60
HORAS_ALIMENTO = [7.5, 14.5]           # alimentación a las 7:30 y 14:30 h
MESES_EXCLUIR = ["2022-07", "2026-01"] # meses incompletos (inicio y fin del historial)

def min_desde_alimento(ts):
    """Minutos transcurridos desde el último horario de alimentación (igual que en Data Understanding)."""
    h = ts.dt.hour + ts.dt.minute / 60
    ultimo = np.select([h >= HORAS_ALIMENTO[1], h >= HORAS_ALIMENTO[0]],
                       [HORAS_ALIMENTO[1], HORAS_ALIMENTO[0]], default=HORAS_ALIMENTO[1] - 24)
    return ((h - ultimo) * 60).round(1)

def temporada_de_mes(m):
    """Estación del año según el mes."""
    if m in (12, 1, 2):
        return "invierno"
    if m in (3, 4, 5):
        return "primavera"
    if m in (6, 7, 8):
        return "verano"
    return "otoño"


# Preprocesamiento para la matriz de correlación

Hay algunas columnsa con que no son numéricas. Por lo que es necesario transformarlas en algo que facilite la lectura y poder representarlas en la matríz de correlación.

Estas columnas son: duracion, patada, incompleto, pezones_no_encontrados, destino_leche

In [ ]:
df = prodLecheSesion.drop(columns=["rcs_celulas_ml", "usuario", "dia_semana"]).copy()
display(df.head())
print("Total de filas", len(df))
print(df.info())

In [ ]:
# ==========================================
# 1. Transformación de eventos de pezones
# Genera exclusivamente la columna: {prefijo}_cant
# ==========================================
pezones_clave = ['TD', 'DI', 'TI', 'DD']

def procesar_evento_pezones(df, columna, prefijo, desglosar_pezones=False):
    """
    Procesa columnas con combinaciones de pezones (ej. 'TD, TI', 'Todos', etc.)
    Crea:
      - {prefijo}_cant: Cantidad de pezones afectados (0 a 4)
      - Opcionalmente {prefijo}_dd, {prefijo}_di, {prefijo}_td, {prefijo}_ti (0 o 1) si desglosar_pezones=True
    """

    def analizar_registro(val):
        # Casos nulos o vacíos
        if pd.isna(val):
            return 0, {p: 0 for p in pezones_clave}

        val_str = str(val).upper().strip()

        # Casos explícitos de negatividad
        if val_str in ['NO', 'FALSE', '0', 'NONE', 'N/A', '']:
            return 0, {p: 0 for p in pezones_clave}

        # Caso "Todos"
        if 'TODOS' in val_str:
            return 4, {p: 1 for p in pezones_clave}

        # Evaluar presencia individual de cada pezón
        presencia = {p: (1 if p in val_str else 0) for p in pezones_clave}
        cant = sum(presencia.values())

        # Si tenía texto válido no nulo pero no coincidió con siglas exactas, se asume 1
        if cant == 0 and len(val_str) > 0:
            cant = 1

        return cant, presencia

    df_out = df.copy()

    # Aplicar la lógica
    resultados = df_out[columna].apply(analizar_registro)

    # Asignar la cantidad total
    df_out[f"{prefijo}_cant"] = [r[0] for r in resultados]

    # Si se solicitó el desglose, agregar las 4 columnas de cada pezón
    if desglosar_pezones:
        for p in pezones_clave:
            p_lower = p.lower()
            df_out[f"{prefijo}_{p_lower}"] = [r[1][p] for r in resultados]

    return df_out

# ==========================================
# 2. Transformación de 'destino_leche'
# ==========================================
# Tanque => 0 | Drenaje / Divert 3 => 1
def procesar_destino_leche(df):
  df['desviacion_leche'] = df['destino_leche'].apply(
      lambda x: 0 if str(x).strip() == 'Tanque' else 1
  )
  return df

Transformar las columnas categóricas en numérico.

In [ ]:
s = df.dropna(subset=["hora_inicio", "ms", "duracion_min"]).sort_values(["ms", "hora_inicio"]).copy()
s["fin"] = s["hora_inicio"] + pd.to_timedelta(s["duracion_min"], unit="m")
s["siguiente_inicio"] = s.groupby("ms")["hora_inicio"].shift(-1)
s["tiempo_libre_min"] = (s["siguiente_inicio"] - s["fin"]).dt.total_seconds() / 60

# Valores negativos = traslapes (posibles errores de registro o duplicados)
print(f"Traslapes (tiempo libre < 0): {(s['tiempo_libre_min'] < 0).sum():,}")
s = s[s["tiempo_libre_min"].between(0, 24 * 60)]
s["posible_cola"] = s["tiempo_libre_min"] <= UMBRAL_COLA_MIN

print(f"Ordeños seguidos de otro en ≤ {UMBRAL_COLA_MIN} min: {s['posible_cola'].mean():.1%}")
s["tiempo_libre_min"].describe(percentiles=[.1, .25, .5, .75, .9]).round(2)

df = s

In [ ]:
df_prep = df.copy()

df_prep = procesar_evento_pezones(df_prep, columna='patada', prefijo='patada')
df_prep = procesar_evento_pezones(df_prep, columna='incompleto', prefijo='incompleto')
df_prep = procesar_evento_pezones(df_prep, columna='pezones_no_encotnrados', prefijo='pezones_no_encontrados')
df_prep = procesar_destino_leche(df_prep)
df_prep["posible_cola"] = df_prep["posible_cola"].astype("Int64")
df_prep = df_prep.drop(columns=["patada", "incompleto", "pezones_no_encotnrados"])

df_prep['es_alta_produccion'] = (
    df_prep['produccion_kg'] > df_prep['produccion_kg'].quantile(0.75)
).astype(int)
df_prep['intervalo_corto'] = (df_prep['intervalo_min'] < 360).astype(int)


pd.set_option('display.max_columns', None)

# Matriz de correlación normal

No se encuentra ninguna variable que esté relacionada con la formación de las colas en general.
La aproximación ingenua de graficar la correlación de todas las variables entre sí para detectar una posible relación entre la formación de las colas con los demás factores no fue exitosa.

Este demostró que las variables en sí no tienen mucha relación general entre ellas y sobre todo que la relación con la formación de las colas, utilizando esta como un buleano de si existe o no la cola no nos mostró ningún dato relevante, puesto que la única relación significativa era con una variable mecánica.

In [ ]:
df_num = df_prep.select_dtypes(include=[np.number])
df_other = df_prep.select_dtypes(exclude=[np.number])

# Escalar minmax
scaler_std = StandardScaler()
df_norm = pd.DataFrame(
  scaler_std.fit_transform(df_num),
  columns=df_num.columns,
  index=df_num.index
)

matriz_corr = df_norm.corr(method='pearson')

plt.figure(figsize=(10, 8))
sns.heatmap(
  matriz_corr,
  annot=True,
  cmap='coolwarm',
  fmt=".2f",
  vmin=-1, vmax=1,
  linewidths=0.5
)
plt.title('Matriz de Correlación')
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------------------
# 1. ANÁLISIS ENFOCADO: Correlación directa contra 'posible_cola'
# ------------------------------------------------------------------------------
# Extraer la serie de correlaciones respecto a 'posible_cola'
corr_posible_cola = matriz_corr[['posible_cola']].copy()

# Eliminar la autocorrelación de 'posible_cola' consigo misma
corr_posible_cola = corr_posible_cola.drop(index='posible_cola').sort_values(by='posible_cola', ascending=False)

# Graficar en barras horizontales
plt.figure(figsize=(8, 10))
sns.barplot(
    x=corr_posible_cola['posible_cola'],
    y=corr_posible_cola.index,
    palette='vlag'
)

plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.title('Correlación de Variables respecto a "posible_cola"', fontsize=13, fontweight='bold')
plt.xlabel('Coeficiente de Correlación (Pearson)')
plt.ylabel('Variables')
plt.grid(axis='x', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------------------------
# 2. TOP 5 FACTORES POSITIVOS Y NEGATIVOS (En texto/DataFrame)
# ------------------------------------------------------------------------------
top_positivos = corr_posible_cola.head(5)
top_negativos = corr_posible_cola.tail(5)

print("=== Top 5 Variables con Mayor Correlación Positiva ===")
display(top_positivos)

print("\n=== Top 5 Variables con Mayor Correlación Negativa ===")
display(top_negativos)


# ------------------------------------------------------------------------------
# 3. MATRIZ COMPLETA LEGIBLE (Triangular con tamaño de fuente ajustado)
# ------------------------------------------------------------------------------
plt.figure(figsize=(18, 14))

# Máscara para ocultar la mitad duplicada
mask = np.triu(np.ones_like(matriz_corr, dtype=bool))

sns.heatmap(
    matriz_corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    vmin=-1, vmax=1,
    linewidths=0.5,
    annot_kws={"size": 7.5},  # Texto interior pequeño para evitar traslapes
    cbar_kws={"shrink": 0.8}
)

plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.title('Matriz de Correlación Completa (Ajustada)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Matriz de correlación - datos preprocesados

Al no encontrar relación en la matriz general de los datos ampliamos la manera de definir una cola a porcentage de formación de cola en lugar de solamente la existencia de la misma. Esto nos brinda mucha más información para el análisis y procesado de datos.

Separamos la información en bloques de 30 minutos, y en base a esto calculamos el porcentaje de creación de las colas.

Los porcentajes de cola van de acuerdo con las tendencias de utilización de la máquina y ordeños horarios.

> **Nota:** el `df_sistema` de esta sección es exploratorio. Se construye desde `prodLecheSesion` (sin la limpieza) y sirvió para decidir las variables y los subconjuntos. La versión definitiva se construye en la sección **Construir los datos**, a partir de `produccion_clean`.

In [ ]:
df_prep.head(3)

In [ ]:
# 1. Crear el bloque de tiempo (30 minutos)
df_prep['bloque_tiempo'] = pd.to_datetime(df_prep['hora_inicio']).dt.floor('30min')

# 2. Agrupar a nivel SISTEMA (Robot + Bloque de tiempo)
df_sistema = df_prep.groupby(['ms', 'bloque_tiempo']).agg(
    # Variable objetivo continua (% de ordeños con cola en el bloque)
    pct_posible_cola=('posible_cola', 'mean'),

    # Métricas de tráfico y duraciones en minutos
    total_ordenios=('id_vaca', 'count'),
    produccion_promedio=('produccion_kg', 'mean'),
    duracion_promedio_min=('duracion_min', 'mean'),
    duracion_total_min=('duracion_min', 'sum'),

    # Acumulación de fallas/retrasos en la ventana
    patadas_totales=('patada_cant', 'sum'),
    incompletos_totales=('incompleto_cant', 'sum'),
    pezones_no_enc_totales=('pezones_no_encontrados_cant', 'sum'),
    desviaciones_leche_totales=('desviacion_leche', 'sum')
).reset_index()

# 3. Métrica de porcentaje de ocupación del robot (30 min disponibles en el bloque)
df_sistema['pct_ocupacion_robot'] = (df_sistema['duracion_total_min'] / 30).clip(upper=1.0)

# 4. Seleccionar columnas numéricas y calcular correlación
df_sistema_num = df_sistema.select_dtypes(include=[np.number])
matriz_corr_sistema = df_sistema_num.corr(method='pearson')

In [ ]:
# ==============================================================================
# 2. GRÁFICA DE BARRAS: % DE COLAS POR HORA DEL DÍA
# ==============================================================================
# Extraer la hora (0-23) desde el bloque de tiempo
df_sistema['hora_del_dia'] = df_sistema['bloque_tiempo'].dt.hour

# Agrupar por hora para obtener el porcentaje promedio de colas
colas_por_hora = df_sistema.groupby('hora_del_dia')['pct_posible_cola'].mean().reset_index()
colas_por_hora['pct_posible_cola'] = colas_por_hora['pct_posible_cola'] * 100  # Convertir a %

plt.figure(figsize=(12, 5))
ax = sns.barplot(
    data=colas_por_hora,
    x='hora_del_dia',
    y='pct_posible_cola',
    palette='Blues_d'
)

plt.title('Porcentaje Promedio de Formación de Colas según la Hora del Día', fontsize=13, fontweight='bold')
plt.xlabel('Hora del Día (00:00 - 23:00)', fontsize=10)
plt.ylabel('% Promedio de Posible Cola', fontsize=10)
plt.grid(axis='y', linestyle=':', alpha=0.7)

# Añadir valores numéricos sobre cada barra
for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f"{p.get_height():.1f}%",
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom',
                    fontsize=8, color='black', xytext=(0, 2),
                    textcoords='offset points')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.colors as mcolors

def bar_corr_plot(matriz_corr_sistema):
  corr_pct_cola = (
      matriz_corr_sistema[['pct_posible_cola']]
      .drop(index='pct_posible_cola')
      .sort_values(by='pct_posible_cola', ascending=False)
  )

  # 1. Definir rango simétrico alrededor de 0 para centrar el gradiente neutro
  max_abs = max(
      abs(corr_pct_cola['pct_posible_cola'].min()),
      abs(corr_pct_cola['pct_posible_cola'].max()),
  )
  norm = mcolors.Normalize(vmin=-max_abs, vmax=max_abs)

  # 2. Elegir mapa de colores divergente (Azul -> Blanco/Neutro -> Rojo)
  cmap = plt.get_cmap('coolwarm')  # También puedes usar 'RdBu_r' o 'vlag'

  # 3. Generar la lista de colores según la intensidad de cada valor
  colores = [cmap(norm(val)) for val in corr_pct_cola['pct_posible_cola']]

  plt.figure(figsize=(8, 6))
  sns.barplot(
      x=corr_pct_cola['pct_posible_cola'], y=corr_pct_cola.index, palette=colores
  )
  plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
  plt.title(
      'Correlación de Variables con el "% de Posible Cola" (30 min)',
      fontsize=12,
      fontweight='bold',
  )
  plt.xlabel('Coeficiente de Correlación de Pearson')
  plt.grid(axis='x', linestyle=':', alpha=0.6)
  plt.tight_layout()
  plt.show()

In [ ]:
# ==============================================================================
# 3. MATRIZ DE CORRELACIÓN COMPLETA
# ==============================================================================
def plot_corr_matrix(matriz_corr_sistema):
  plt.figure(figsize=(10, 8))
  mask = np.triu(np.ones_like(matriz_corr_sistema, dtype=bool))

  sns.heatmap(
      matriz_corr_sistema,
      mask=mask,
      annot=True,
      fmt=".2f",
      cmap='coolwarm',
      vmin=-1, vmax=1,
      linewidths=0.5,
      annot_kws={"size": 9}
  )
  plt.title('Matriz de Correlación Agregada por Sistema / Ventanas de Tiempo', fontsize=13, fontweight='bold')
  plt.tight_layout()
  plt.show()

En las siguientes gráficas se ven las variables generales y su nivel de relación con la formación de las colas.
En este caso se ve como pct-ocupación, duración total de ordeños y total ordeños tienen relación muy fuerte con la formación de las colas, al igual que desviación de la leche es la única con una tendencia negativa.

In [ ]:
bar_corr_plot(matriz_corr_sistema)
plot_corr_matrix(matriz_corr_sistema)

Al tomar en cuenta la temporalidad podemos observar patrones que ya habíamos visto anteriormente con el porcentaje de ocupación, hay ciertas horas donde el porcentaje de colas aumentará o disminuirá al igual que ver que en temporadas de invierno la cola es más propensa a aumentar. Pero todo esto tienen magnitudes muy pequeñas.

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# ==============================================================================
# 1. CREAR VARIABLES DE HORA Y TEMPORADA (ONE-HOT ENCODING)
# ==============================================================================
# A. Horas del día (00:00 a 23:00)
df_prep['hora_str'] = df_prep['hora'].apply(lambda x: f'hora_{x:02d}h')
dummies_horas = pd.get_dummies(df_prep['hora_str'], prefix='', prefix_sep='')

# B. Temporadas del año
def obtener_temporada(mes):
    m = mes.month if hasattr(mes, 'month') else int(mes)
    if m in [12, 1, 2]:
        return 'temp_Invierno'
    elif m in [3, 4, 5]:
        return 'temp_Primavera'
    elif m in [6, 7, 8]:
        return 'temp_Verano'
    else:
        return 'temp_Otono'

df_prep['temporada'] = df_prep['fecha'].apply(obtener_temporada)
dummies_temporadas = pd.get_dummies(df_prep['temporada'])

# Unir con la variable objetivo convertible a float
df_encoded = pd.concat([
    df_prep[['posible_cola']].astype(float),
    dummies_horas,
    dummies_temporadas
], axis=1)

# ==============================================================================
# 2. CORRELACIÓN 1: HORAS DEL DÍA VS. FORMACIÓN DE COLA
# ==============================================================================
corr_horas = (
    df_encoded[['posible_cola'] + list(dummies_horas.columns)]
    .corr(method='pearson')[['posible_cola']]
    .drop(index='posible_cola')
)

# Graficar
max_abs_h = max(abs(corr_horas['posible_cola'].min()), abs(corr_horas['posible_cola'].max()))
norm_h = mcolors.Normalize(vmin=-max_abs_h, vmax=max_abs_h)
colores_h = [plt.get_cmap('coolwarm')(norm_h(val)) for val in corr_horas['posible_cola']]

plt.figure(figsize=(10, 6))
sns.barplot(x=corr_horas['posible_cola'], y=corr_horas.index, palette=colores_h)
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.title('Correlación por Hora del Día (One-Hot Encoding) vs "% Posible Cola"', fontweight='bold')
plt.xlabel('Coeficiente de Correlación de Pearson')
plt.grid(axis='x', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

# ==============================================================================
# 3. CORRELACIÓN 2: TEMPORADA DEL AÑO VS. FORMACIÓN DE COLA
# ==============================================================================
corr_temp = (
    df_encoded[['posible_cola'] + list(dummies_temporadas.columns)]
    .corr(method='pearson')[['posible_cola']]
    .drop(index='posible_cola')
    .sort_values(by='posible_cola', ascending=False)
)

# Graficar
max_abs_t = max(abs(corr_temp['posible_cola'].min()), abs(corr_temp['posible_cola'].max()))
norm_t = mcolors.Normalize(vmin=-max_abs_t, vmax=max_abs_t)
colores_t = [plt.get_cmap('coolwarm')(norm_t(val)) for val in corr_temp['posible_cola']]

plt.figure(figsize=(7, 4))
sns.barplot(x=corr_temp['posible_cola'], y=corr_temp.index, palette=colores_t)
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.title('Correlación por Temporada del Año (One-Hot Encoding) vs "% Posible Cola"', fontweight='bold')
plt.xlabel('Coeficiente de Correlación de Pearson')
plt.grid(axis='x', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

## Criterios de selección de variables

Las variables que vamos a seleccionar deben tener algún tipo de relación con nuestra variable de estudio, que es la formación de las colas. Hay que dar preferencia a variables con un alto grado de relación, pero dado el resultado de que varias variables tienen un nivel poco óptimo en su correlación con la formación de las colas este criterio baja un poco.

Variable dependiente
- posible_cola

Variables de alta relación

1. pct_ocupacion_robot
1. duracion_total_min
1. total_ordenios

Variables de baja relación que convendría incluir:

1. patadas
1. produccion_promedio
1. desviacion_leche_totales

Variables temporales

1. hora
1. temporada

## Seleccion de subconjutos y muestreo


---




Dentro de esta sección se definen los subconjuntos de datos que serán utilizados para el modelado y se evaluará si es necesario aplicar un muestreo. Cabe mencionar que el propósito de esta área es quitar los datos que distorsionan la medición de las colas antes de construir el modelo.

La matriz de correlación elaborada en la sección anterior mostró que a nivel ordeño individual ninguna variable se relaciona con la formación de las colas. Sin embargo, al agrupar por bloques de 30 minutos podemos observar que en realidad sí aparecen algunas relaciones. Es debido a eso que los subconjuntos se definen sobre esos bloques, revisando los siguientes puntos:

1. Que robots y meses tienen datos
2. Que dias tienen operación incompleta
3. Que bloques de 30 min tienen tan poco ordeños que su % de cola no es confiable
4. Si hace falta muestreo, y cómo dividir los datos para el modelo.
5. Si hace falta muestreo y cómo dividir los datos para el modelado.


Cabe mencionar que, los umbrales están como constantes en mayúsculas al inicio de cada celda para poder ajustarlos.


### Cobertura por robot y por mes


---
Los meses con caídas bruscas o en cero indican periodos sin registro o un robot fuera de servicio.



In [ ]:
print("Ordeños por robot:")
display(df_prep['ms'].value_counts().to_frame('ordenios'))

cobertura_mensual = (
    df_prep.assign(mes_periodo=df_prep['hora_inicio'].dt.to_period('M'))
    .groupby(['mes_periodo', 'ms']).size()
    .unstack('ms', fill_value=0)
)
display(cobertura_mensual)

ax = cobertura_mensual.plot(figsize=(14, 4), marker='o', markersize=3)
ax.set_title('Ordeños por mes y robot', fontweight='bold')
ax.set_xlabel('Mes')
ax.set_ylabel('Ordeños')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Dias con operacion incompleta


---
Si un día tiene registros faltantes, el tiempo libre min entre ordeños se calcula sobre huecos que en realidad no existieron, el robot aparece ser libre y el ordeño se marca como sin cola aunque si la hubo, es por eso que los días según la variable objetivo.

En este caso un día se marca como incompleto si tiene menos del umbral del día x la mediana móvil de ordeños diarios de ese robot.




In [ ]:
UMBRAL_DIA = 0.5     # el día debe tener al menos 50 % de los ordeños de un día típico
VENTANA_DIAS = 30    # días con registro que se usan para calcular la mediana móvil

diario = (
    df_prep.assign(dia=df_prep['hora_inicio'].dt.normalize())
    .groupby(['ms', 'dia']).size().rename('ordenios_dia').reset_index()
    .sort_values(['ms', 'dia'])
)
diario['mediana_movil'] = (
    diario.groupby('ms')['ordenios_dia']
    .transform(lambda x: x.rolling(VENTANA_DIAS, min_periods=7, center=True).median())
    .fillna(diario.groupby('ms')['ordenios_dia'].transform('median'))
)
diario['dia_completo'] = diario['ordenios_dia'] >= UMBRAL_DIA * diario['mediana_movil']

# Días del calendario sin ningún registro
for robot, g in diario.groupby('ms'):
    rango = pd.date_range(g['dia'].min(), g['dia'].max(), freq='D')
    faltantes = rango.difference(g['dia'])
    print(f"{robot}: {len(faltantes)} días sin ningún registro entre {rango[0].date()} y {rango[-1].date()}")

resumen_dias = diario.groupby('ms')['dia_completo'].agg(dias_con_registro='size', dias_completos='sum')
resumen_dias['pct_incompletos'] = (1 - resumen_dias['dias_completos'] / resumen_dias['dias_con_registro']).mul(100).round(2)
display(resumen_dias)

print("Ejemplos de días incompletos:")
display(diario.loc[~diario['dia_completo']].head(15))

### Confiabilidad del % de cola según los ordeños del bloque


---

En un bloque con un ordeño, solo puede valor 0% o 100%, por lo que es una medida demasiado ruidosa, es por eso que se elaboró una tabla en donde muestra cuántos bloques hay con cada número de ordeños y que tan disperso es su % de cola.  




In [ ]:
dist_bloques = (
    df_sistema.groupby('total_ordenios')['pct_posible_cola']
    .agg(bloques='size', pct_cola_prom='mean', desv_est='std')
)
dist_bloques['pct_bloques'] = (dist_bloques['bloques'] / dist_bloques['bloques'].sum() * 100).round(2)
display(dist_bloques.round(3))

MIN_ORDENIOS_BLOQUE = 1

### Las horas con menos cola coinciden con lavados de robot?


---

En la gráfica por hora hay caídas, si estas coinciden con programa lavado, el robot no está disponible en esas horas. En este caso no conviene excluir esos bloques debido a que el lavado puede ser causa de la cola, es por eso que es mejor marcarlos como atributos en Construir los datos.



In [ ]:
lavado_por_hora = (
    df_prep.assign(con_lavado=df_prep['programa_lavado'].notna())
    .groupby('hora')['con_lavado'].mean().mul(100)
)

fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.bar(colas_por_hora['hora_del_dia'], colas_por_hora['pct_posible_cola'],
        color='steelblue', alpha=0.6, label='% posible cola')
ax1.set_xlabel('Hora del día')
ax1.set_ylabel('% promedio de posible cola')
ax1.set_xticks(range(24))

ax2 = ax1.twinx()
ax2.plot(lavado_por_hora.index, lavado_por_hora.values,
         color='darkred', marker='o', label='% ordeños con programa de lavado')
ax2.set_ylabel('% de ordeños con programa de lavado')

fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.95))
plt.title('Formación de colas vs. lavados del robot por hora', fontweight='bold')
plt.tight_layout()
plt.show()

### Aplicar los subconjuntos



---

Se aplican los criterios en orden y se registra cuántos bloques quita cada uno. La tabla resultante sirve como evidencia para la base lógica de inclusión / exclusión


In [ ]:
df_sistema_sel = df_sistema.copy()
df_sistema_sel['dia'] = df_sistema_sel['bloque_tiempo'].dt.normalize()
df_sistema_sel = df_sistema_sel.merge(diario[['ms', 'dia', 'dia_completo']], on=['ms', 'dia'], how='left')

registro = [('Bloques iniciales', len(df_sistema_sel))]

df_sistema_sel = df_sistema_sel[df_sistema_sel['dia_completo'].eq(True)]
registro.append(('Quitar días con operación incompleta', len(df_sistema_sel)))

df_sistema_sel = df_sistema_sel[df_sistema_sel['total_ordenios'] >= MIN_ORDENIOS_BLOQUE]
registro.append((f'Quitar bloques con menos de {MIN_ORDENIOS_BLOQUE} ordeños', len(df_sistema_sel)))

df_sistema_sel = df_sistema_sel.drop(columns=['dia', 'dia_completo']).reset_index(drop=True)

tabla_seleccion = pd.DataFrame(registro, columns=['paso', 'bloques'])
tabla_seleccion['quitados'] = (-tabla_seleccion['bloques'].diff()).fillna(0).astype(int)
tabla_seleccion['pct_restante'] = (tabla_seleccion['bloques'] / tabla_seleccion['bloques'].iloc[0] * 100).round(1)
display(tabla_seleccion)

# ¿Se mantienen las relaciones después de filtrar?
bar_corr_plot(df_sistema_sel.select_dtypes(include=[np.number]).corr(method='pearson'))

### Hace falta muestreo?


---

Debido a que son datos de series de tiempo no se deben de hacer un muestreo aleatorio ni un k-fold normal para validarlo. De hecho si se llegan a mezclar los bloques en el entrenamiento el modelo ve el periodo que se quiere predecir. En este caso se utiliza TimeSeriesSplit el cual funciona para entrenar con el pasado y prueba con el siguiente periodo.


In [ ]:
from sklearn.model_selection import TimeSeriesSplit

mem_mb = df_sistema_sel.memory_usage(deep=True).sum() / 1024**2
print(f"Dataset seleccionado: {len(df_sistema_sel):,} bloques, {mem_mb:.1f} MB en memoria")
print(f"Balance por ordeño (posible_cola): {df_prep['posible_cola'].mean():.1%} de ordeños con posible cola")
print("\nDistribución de pct_posible_cola por bloque:")
print(df_sistema_sel['pct_posible_cola'].describe().round(3))

N_FOLDS = 5
df_sistema_sel = df_sistema_sel.sort_values(['bloque_tiempo', 'ms']).reset_index(drop=True)
tscv = TimeSeriesSplit(n_splits=N_FOLDS)

folds = []
for i, (idx_ent, idx_prueba) in enumerate(tscv.split(df_sistema_sel), start=1):
    t = df_sistema_sel['bloque_tiempo']
    folds.append({
        'fold': i,
        'entrenamiento': f"{t.iloc[idx_ent[0]].date()} a {t.iloc[idx_ent[-1]].date()}",
        'prueba': f"{t.iloc[idx_prueba[0]].date()} a {t.iloc[idx_prueba[-1]].date()}",
        'bloques_entrenamiento': len(idx_ent),
        'bloques_prueba': len(idx_prueba),
    })
display(pd.DataFrame(folds))

### Base lógica: subconjuntos y muestreo

Subconjuntos incluidos
- Bloques de 30 min por robot, porque el objetivo es la formación de colas a nivel sistema y a nivel ordeño no se encontró relación.
- Solo dias con operación completa: se excluyen 7 días VMS1:2, VMS2:1 y BMS 3:4, los cuales equivalen a 50 bloques o el 0.04% del total, es debido a eso que los registros faltantes inflan el tiempo libre y marcan de manera false como sin cola. Como se puede analizar los tres robots tienen un registro todos los días entre 2022-07-13 y 2026-01-28, los días excluidos son días parciales como es el ejemplo de 2026-01-28.
- Se conservan los bloques con 1 ordeño, o sea el 35.8% del total. Aunque su % de cola solo puede ser 0 o 100, se tiene una desviación estándar de 0.332, lo cual es casi igual a la de los bloques que tienen dos ordeños, la cual es de 0.320, y estos representan periodos de baja actividad con poca cola, alrededor de un 12.7% en promedio. Al excluir esto causaría un sesgo a la variable objetivo hacia los periodos con más cola.  
- Solo meses completos: se excluyen julio 2022 y enero 2026, que son el inicio y el fin del historial y no están completos. Equivalen a 4,552 bloques o el 3.3% del total.


Subconjuntos no excluidos, se convierten en atributos
- Horas con lavado del robot, debido a que coinciden con las caídas de la cola, esto se puede ver como los dos picos de lavado a las 4 hrs hay un 7% de ordeños con programa de lavado y a las 20hrs,  un 6.9%. Estas son justo las horas con menos cola siendo de 28.7% y 33.4% respectivamente. En este caso no se excluyen debido a que el lavado puede ser parte de la causa de las colas, es por eso que se marcan como atributo en el área de construir los datos.


Muestreo
- No se requiere balanceo debido a que el 53.7% de los ordeños con posible cola ya están balanceadas.
- No requieren muestreo por tamaño debido a que el dataset final (df_sistema_sel, sección Construir los datos) tiene 132,027 bloques y Colab los maneja sin problemas.
- La validación se hará con TimeSeriesSplit de 5 folds de 22,004 bloques de prueba cada uno, así entrenando siempre con el pasado y probando con el periodo siguiente.


# Limpiar datos

**Política de limpieza y ruido:** se corrigen inconsistencias de integridad y se auditan anomalías, los eventos reales atípicos se conservan. Los registros con mediciones productivas sospechosas permanecen para el análisis temporal, pero su producción se omite en el promedio por bloque. Los conflictos temporales se excluyen conservando evidencia, especialmente cuando la hora local es ambigua.

1. **Columnas vacías o sin información**

<!-- -->

1. Eliminar alimento (constante)
1. Para empezar, se eliminaron las columnas rcs_celulas_ml y usuario, ya que están vacías.
1. Convertir programa_lavado en boolean, debido a que no hay diferencia en valores
1. razon_desviación: Transformar nulos cuando si hubo desviación en "sin razón registrada"


In [ ]:
# Columnas vacías o sin información
consumo_clean = concentradoConsumido.copy()
produccion_clean = prodLecheSesion.copy()

# print("Tamaño de valores diferentes en consumo", len(consumo_clean[consumo_clean["consumo"] != consumo_clean["consumido"]]))
consumo_clean.drop(columns="consumido", inplace=True)

produccion_clean.drop(columns=["rcs_celulas_ml", "usuario"], inplace=True)
produccion_clean["programa_lavado"] = produccion_clean["programa_lavado"].eq("Local Aclarado").astype("boolean")

condicion = (produccion_clean["razon_desviacion"].isna()) & (produccion_clean["destino_leche"] != "Tanque")
produccion_clean.loc[condicion, "razon_desviacion"] = "sin razón registrada"

2. **Valores faltantes por cuarto y MDI**
<!-- -->
1. Los valores faltantes por cuarto se dejarán como nulos.
1. Mantener valores de MDI faltnates como NAN
<!-- -->
3. **Incidencias con significado ambiguo**
<!-- -->
1. Remplazar 0s de conductividad y media_flujo por NaN
1. Se añadió las columnas / flag de "patadas_cant" / "incompletos_cant", "pezones_no_encontrados_cant". En lugar de solo tener positivo o negativo tenemos cantidad de incidencias, aunque solo captura 1 por pezón, al igual que las columnas de donde ocurrió la insidencia.

In [ ]:
produccion_clean = procesar_evento_pezones(produccion_clean, columna='patada', prefijo='patada', desglosar_pezones=True)
produccion_clean = procesar_evento_pezones(produccion_clean, columna='incompleto', prefijo='incompleto', desglosar_pezones=True)
produccion_clean = procesar_evento_pezones(produccion_clean, columna='pezones_no_encotnrados', prefijo='pezones_no_encontrados', desglosar_pezones=True)

# cond = (produccion_clean["patada_cant"] != 0) & (produccion_clean["incompleto_cant"] != 0) & (produccion_clean["pezones_no_encontrados_cant"] != 0)

# display(produccion_clean.loc[cond].head(5))

In [ ]:
# Conductividad y media de flujo en 0 no tienen sentido físico: se reemplazan por NaN
cols_ceros = [c for c in produccion_clean.columns if c.startswith(("conductividad_", "media_flujo_"))]
print("Valores en 0 que se reemplazan por NaN:")
display((produccion_clean[cols_ceros] == 0).sum().to_frame("ceros"))
produccion_clean[cols_ceros] = produccion_clean[cols_ceros].replace(0, np.nan)

# destino_leche: un nulo no se debe contar como leche desviada (ver desviacion_leche en Construir los datos)
print("Valores de destino_leche:")
display(produccion_clean["destino_leche"].value_counts(dropna=False).to_frame("ordeños"))


4. **Valores extremos o imposibles**

1. Se conservaron los ordeños de duración atípica, identificados con `ordenio_anomalo` cuando duran menos de 1 o más de 30 minutos.
2. Se identificaron diferencias entre la producción total y la suma de los cuatro cuartos cuando las cuatro mediciones están disponibles. Se utiliza una tolerancia absoluta de 0.1 kg y se conserva la bandera `desbalance_produccion` sin eliminar registros.
3. Se mantuvo MDI sin modificaciones.

In [ ]:
# Identificar duraciones atípicas sin eliminar los ordeños
cond = (produccion_clean["duracion_min"] < 1) | (produccion_clean["duracion_min"] > 30)
produccion_clean["ordenio_anomalo"] = cond.astype(int)

# Aplicar el mismo criterio de desbalance en limpieza y auditoría
cuartos = ["produccion_di", "produccion_dd", "produccion_ti", "produccion_td"]
suma_cuartos = produccion_clean[cuartos].sum(axis=1, min_count=4)
produccion_clean["desbalance_produccion"] = (
    suma_cuartos.notna()
    & produccion_clean["produccion_kg"].notna()
    & ~np.isclose(produccion_clean["produccion_kg"], suma_cuartos, atol=0.1, rtol=0)
)

print("Ordeños con duración atípica:", int(produccion_clean["ordenio_anomalo"].sum()))
print("Desbalances de producción conservados:", int(produccion_clean["desbalance_produccion"].sum()))

5. **Se eliminaron las inconsistencias entre las columnas**

1. Se eliminaron los dos registros sin máquina de ordeño asociada, que contenían valores negativos en otras columnas.
2. Se verificó la consistencia temporal por máquina utilizando el inicio y la duración de cada ordeño, sin limitar la detección a un mismo día. En caso de solapamiento se conserva el registro de inicio más temprano y se excluye el siguiente que entra en conflicto con el último registro conservado. Esta regla evita descartar registros adicionales por compararlos con ordeños que ya fueron excluidos. Los registros descartados se guardan en `traslapes` para auditoría.

In [ ]:
print("Antes del filtro - quitar ordeños que no tienen máquina: ", len(produccion_clean[produccion_clean["ms"].isna()]))
produccion_clean.dropna(subset=["ms"], inplace=True)
print("Después del filtro", len(produccion_clean[produccion_clean["ms"].isna()]))

In [ ]:
# Ordenar de manera estable los ordeños por máquina e inicio
produccion_clean = produccion_clean.sort_values(["ms", "hora_inicio"], kind="mergesort").copy()
produccion_clean["hora_fin"] = (
    produccion_clean["hora_inicio"]
    + pd.to_timedelta(produccion_clean["duracion_min"], unit="m")
)

# Comparar cada ordeño únicamente contra el último que se decidió conservar.
# Ante un conflicto se conserva el que comenzó primero (desempate: orden original).
indices_excluidos = []
minutos_traslape = {}
for maquina, grupo in produccion_clean.groupby("ms", sort=False):
    fin_ultimo_conservado = None
    for indice, fila in grupo.iterrows():
        inicio, fin = fila["hora_inicio"], fila["hora_fin"]
        if fin_ultimo_conservado is not None and inicio < fin_ultimo_conservado:
            indices_excluidos.append(indice)
            minutos_traslape[indice] = (fin_ultimo_conservado - inicio).total_seconds() / 60
        else:
            fin_ultimo_conservado = fin

# Conservar una copia de los registros excluidos para la auditoría
traslapes = produccion_clean.loc[indices_excluidos].copy()
traslapes["traslape_min"] = traslapes.index.map(minutos_traslape)
print("Ordeños excluidos por solapamiento:", len(traslapes))
display(traslapes[["id_vaca", "ms", "hora_inicio", "hora_fin", "traslape_min"]])

produccion_clean = produccion_clean.drop(index=indices_excluidos).copy()
produccion_clean.drop(columns=["hora_fin"], inplace=True)
print("Registros restantes:", len(produccion_clean))

6. **Cobertura y sesgo del historial**
<!--  -->
1. Exclusión de julio 2022 y enero 2026 para solo tener información completa de meses
1. Las vacas deben tener por lo menos un historial de 30 días

In [ ]:
produccion_clean_strict = produccion_clean.copy()

# Asegurar formato datetime por si acaso
produccion_clean['fecha'] = pd.to_datetime(produccion_clean['fecha'])

# Filtrar excluyendo julio-2022 y enero-2026
mask_jul_2022 = (produccion_clean['fecha'].dt.year == 2022) & (
    produccion_clean['fecha'].dt.month == 7
)
mask_ene_2026 = (produccion_clean['fecha'].dt.year == 2026) & (
    produccion_clean['fecha'].dt.month == 1
)

produccion_clean_strict = produccion_clean[
    ~(mask_jul_2022 | mask_ene_2026)
].copy()

In [ ]:
# Convertir fecha a datetime si no lo está
produccion_clean_strict["fecha"] = pd.to_datetime(produccion_clean_strict["fecha"])

# 1. Calcular el rango de días activos por vaca
rango_dias_vaca = (
    produccion_clean_strict.groupby("id_vaca")["fecha"]
    .agg(lambda x: (x.max() - x.min()).days)
)

# 2. Obtener lista de vacas con >= 30 días
vacas_validas = rango_dias_vaca[rango_dias_vaca >= 30].index

# 3. Filtrar el DataFrame
produccion_clean_strict = produccion_clean_strict[
    produccion_clean_strict["id_vaca"].isin(vacas_validas)
].copy()

In [ ]:
display(produccion_clean.head(3))
display(produccion_clean_strict.head(3))

### Auditoría de limpieza
Se verifican errores de integridad y se contabilizan las anomalías conservadas. Los valores extremos no se eliminan automáticamente.


In [ ]:
# Auditoría de integridad sobre los ordeños depurados (sin modificar produccion_clean).
revision = produccion_clean.sort_values(["ms", "hora_inicio"]).copy()
revision["hora_fin_revision"] = (
    revision["hora_inicio"] + pd.to_timedelta(revision["duracion_min"], unit="m")
)
fin_previo = revision.groupby("ms")["hora_fin_revision"].transform(
    lambda serie: serie.cummax().shift()
)
solapamientos_restantes = int((revision["hora_inicio"] < fin_previo).sum())

# Usar exactamente la bandera y el criterio definidos durante la limpieza.
desbalances_revision = int(produccion_clean["desbalance_produccion"].sum())
flujo_revision = (
    produccion_clean["produccion_kg"] / produccion_clean["duracion_min"].replace(0, np.nan)
)
flujos_sospechosos_revision = int(flujo_revision.gt(7).sum())

auditoria_limpieza = pd.DataFrame({
    "control": [
        "Duplicados exactos restantes", "Robots faltantes restantes",
        "Fechas faltantes restantes", "Duraciones no positivas restantes",
        "Solapamientos excluidos", "Solapamientos restantes",
        "Desbalances de producción conservados", "Flujos calculados > 7 kg/min conservados"
    ],
    "registros": [
        int(produccion_clean.duplicated().sum()),
        int(produccion_clean["ms"].isna().sum()),
        int(produccion_clean["hora_inicio"].isna().sum()),
        int(produccion_clean["duracion_min"].le(0).sum()),
        len(traslapes), solapamientos_restantes,
        desbalances_revision, flujos_sospechosos_revision
    ]
})
display(auditoria_limpieza)
print("Ordeños disponibles antes de construir bloques:", len(produccion_clean))
print("Dimensiones:", produccion_clean.shape)

# Los nulos restantes se reportan; no se eliminan automáticamente.
nulos_finales = produccion_clean.isna().sum().sort_values(ascending=False)
display(nulos_finales[nulos_finales.gt(0)].to_frame("nulos"))

assert produccion_clean["ms"].notna().all(), "Persisten robots faltantes"
assert produccion_clean["hora_inicio"].notna().all(), "Persisten fechas faltantes"
assert produccion_clean["duracion_min"].gt(0).all(), "Persisten duraciones inválidas"
assert not produccion_clean.duplicated().any(), "Persisten duplicados exactos"
assert solapamientos_restantes == 0, "Persisten solapamientos temporales"
print("Auditoría de integridad completada. Anomalías productivas conservadas para revisión.")

### Cobertura recalculada después de la limpieza

Se recalcula `diario` con los ordeños limpios **antes** de excluir intervalos sin etiqueta. La cobertura preliminar de la celda 36 es exploratoria y no se utiliza para seleccionar los bloques definitivos. Se conserva el umbral original (50 % de la mediana móvil de 30 días).


In [ ]:
# Cobertura definitiva: usar únicamente ordeños depurados, antes de construir posible_cola.
# Se conserva la cobertura exploratoria calculada previamente con df_prep.
diario_preliminar = diario.copy()

diario = (
    produccion_clean.assign(dia=produccion_clean["hora_inicio"].dt.normalize())
    .groupby(["ms", "dia"])
    .size()
    .rename("ordenios_dia")
    .reset_index()
    .sort_values(["ms", "dia"])
)

# Mantener los mismos parámetros de la evaluación exploratoria.
mediana_robot = diario.groupby("ms")["ordenios_dia"].transform("median")
diario["mediana_movil"] = (
    diario.groupby("ms")["ordenios_dia"]
    .transform(lambda s: s.rolling(VENTANA_DIAS, min_periods=7, center=True).median())
    .fillna(mediana_robot)
)
diario["dia_completo"] = diario["ordenios_dia"] >= UMBRAL_DIA * diario["mediana_movil"]

resumen_cobertura = diario.groupby("ms")["dia_completo"].agg(
    dias_con_registro="size", dias_completos="sum"
)
resumen_cobertura["pct_incompletos"] = (
    100 * (1 - resumen_cobertura["dias_completos"] / resumen_cobertura["dias_con_registro"])
).round(2)
print("Cobertura definitiva calculada con ordeños limpios:")
display(resumen_cobertura)

# Días sin registros: se reportan por separado y no se clasifican como completos.
for robot, grupo in diario.groupby("ms"):
    calendario = pd.date_range(grupo["dia"].min(), grupo["dia"].max(), freq="D")
    dias_ausentes = calendario.difference(grupo["dia"])
    print(f"{robot}: {len(dias_ausentes)} días sin registros en el periodo observado")


# Construir los datos

En esta sección se generan los atributos derivados que se usarán en el modelado. Primero se construyen los atributos por ordeño sobre `produccion_clean`, después se agregan a nivel robot × bloque de 30 min (`df_sistema`) y se agregan los atributos del bloque y los del bloque anterior. Al final se aplican los subconjuntos definidos en la sección de selección.

`df_sistema` se construye desde `produccion_clean` y no desde `produccion_clean_strict`: el filtro de vacas con menos de 30 días de historial es para los análisis por vaca. Esas vacas sí ocuparon el robot, así que quitarlas subestimaría la ocupación y la cola de cada bloque.

No se generan registros nuevos.

## Atributos de cola por ordeño

Para cada ordeño se calcula el tiempo libre del robot hasta el siguiente ordeño. Si es de `UMBRAL_COLA_MIN` minutos o menos se marca como `posible_cola`. También se asigna el bloque de 30 min y el porcentaje de ordeños con posible cola de ese bloque (`pct_cola_bloque`).

In [ ]:
s = produccion_clean.sort_values(["ms", "hora_inicio"]).copy()
s["fin"] = s["hora_inicio"] + pd.to_timedelta(s["duracion_min"], unit="m")
s["siguiente_inicio"] = s.groupby("ms")["hora_inicio"].shift(-1)
s["tiempo_libre_min"] = (s["siguiente_inicio"] - s["fin"]).dt.total_seconds() / 60

# Auditoría ANTES de descartar intervalos sin etiqueta confiable.
exclusiones_intervalos = pd.DataFrame({
    "motivo": ["Tiempo libre negativo", "Tiempo libre mayor a 24 h", "Sin siguiente ordeño"],
    "registros": [int(s["tiempo_libre_min"].lt(0).sum()),
                  int(s["tiempo_libre_min"].gt(1440).sum()),
                  int(s["tiempo_libre_min"].isna().sum())]
})
display(exclusiones_intervalos)
# Solo se construye la etiqueta en intervalos verificables de 0 a 24 h.
s = s[s["tiempo_libre_min"].between(0, 24 * 60)].copy()
s["posible_cola"] = s["tiempo_libre_min"] <= UMBRAL_COLA_MIN

s["bloque_tiempo"] = s["hora_inicio"].dt.floor("30min")

s["pct_cola_bloque"] = (
    s.groupby(["ms", "bloque_tiempo"])["posible_cola"]
    .transform("mean") * 100
)

produccion_clean = s


## Otros atributos por ordeño

- `min_desde_alimento`: minutos desde la última alimentación (7:30 o 14:30 h).
- `desviacion_leche`: 1 si la leche no fue al tanque, 0 si fue al tanque y NaN si no hay destino registrado.
- Ciclo de lactancia (`dias_en_leche`, `etapa_lactancia`, `grupo_vaca` y sus indicadores `es_*`): se traen de S0 (sección 23 de Data Understanding) uniendo por vaca, robot y hora de inicio. Quedan en NaN si no se conoce el inicio de la lactancia.

Al final se actualiza `produccion_clean_strict` para que tenga los mismos atributos.

In [ ]:
produccion_clean["min_desde_alimento"] = min_desde_alimento(produccion_clean["hora_inicio"])

destino = produccion_clean["destino_leche"]
produccion_clean["desviacion_leche"] = np.where(
    destino.isna(), np.nan, (destino.astype(str).str.strip() != "Tanque").astype(float)
)

# Ciclo de lactancia desde S0
LLAVES = ["id_vaca", "ms", "hora_inicio"]
COLS_LACTANCIA = ["dias_en_leche", "etapa_lactancia", "grupo_vaca",
                  "es_primer_parto", "es_fresca", "es_alta", "es_baja", "es_extendida"]

lac_s0 = s0[LLAVES + COLS_LACTANCIA].copy()
lac_s0["hora_inicio"] = lac_s0["hora_inicio"].astype("datetime64[ns]")
lac_s0 = lac_s0.drop_duplicates(subset=LLAVES).set_index(LLAVES)

produccion_clean["hora_inicio"] = produccion_clean["hora_inicio"].astype("datetime64[ns]")
produccion_clean = (produccion_clean.drop(columns=COLS_LACTANCIA, errors="ignore")
                    .join(lac_s0, on=LLAVES))

print(f"Ordeños con etapa de lactancia conocida: {produccion_clean['etapa_lactancia'].notna().mean():.1%}")
print(f"Ordeños sin destino de leche: {produccion_clean['desviacion_leche'].isna().sum():,}")

# produccion_clean_strict con los mismos atributos (mismas filas que antes)
produccion_clean_strict = produccion_clean[produccion_clean.index.isin(produccion_clean_strict.index)].copy()
print(f"produccion_clean: {produccion_clean.shape} | produccion_clean_strict: {produccion_clean_strict.shape}")


## df_sistema a partir de los datos limpios

Se agrupan los ordeños de `produccion_clean` por robot y bloque de 30 min con las mismas agregaciones de la sección de selección, más el lavado y la composición por etapa de lactancia. Las proporciones van de 0 a 1.

In [ ]:
df_sistema = (
    produccion_clean.groupby(["ms", "bloque_tiempo"])
    .agg(
        # Variable objetivo
        pct_posible_cola=("posible_cola", "mean"),
        # Tráfico y duraciones
        total_ordenios=("id_vaca", "count"),
        produccion_promedio=("produccion_kg", "mean"),
        duracion_promedio_min=("duracion_min", "mean"),
        duracion_total_min=("duracion_min", "sum"),
        # Incidencias
        patadas_totales=("patada_cant", "sum"),
        incompletos_totales=("incompleto_cant", "sum"),
        pezones_no_enc_totales=("pezones_no_encontrados_cant", "sum"),
        desviaciones_leche_totales=("desviacion_leche", "sum"),
        # Lavado
        ordenios_con_lavado=("programa_lavado", "sum"),
        # Composición por etapa de lactancia (solo ordeños con etapa conocida)
        pct_primer_parto=("es_primer_parto", "mean"),
        pct_fresca=("es_fresca", "mean"),
        pct_alta=("es_alta", "mean"),
        pct_baja=("es_baja", "mean"),
        pct_extendida=("es_extendida", "mean"),
    )
    .reset_index()
)
df_sistema["pct_ocupacion_robot"] = (df_sistema["duracion_total_min"] / MIN_VENTANA).clip(upper=1.0)

print(f"df_sistema: {len(df_sistema):,} bloques x {df_sistema.shape[1]} columnas")
display(df_sistema.head())


## Atributos por bloque

- `hora`, `dia_semana` y `temporada`: se derivan de `bloque_tiempo`.
- `min_desde_alimento`: minutos desde la última alimentación al inicio del bloque.
- `hubo_lavado`: 1 si algún ordeño del bloque tuvo programa de lavado.
- `vacas_activas_dia`: vacas distintas ordeñadas ese día en los tres robots.
- `ordenios_otros_robots`: ordeños que empezaron en los otros robots en el mismo bloque.

In [ ]:
df_sistema["hora"] = df_sistema["bloque_tiempo"].dt.hour
df_sistema["dia_semana"] = df_sistema["bloque_tiempo"].dt.dayofweek
df_sistema["temporada"] = df_sistema["bloque_tiempo"].dt.month.map(temporada_de_mes)
df_sistema["min_desde_alimento"] = min_desde_alimento(df_sistema["bloque_tiempo"])

df_sistema["ordenios_con_lavado"] = df_sistema["ordenios_con_lavado"].astype(int)
df_sistema["hubo_lavado"] = (df_sistema["ordenios_con_lavado"] > 0).astype(int)

vacas_por_dia = produccion_clean.groupby(produccion_clean["hora_inicio"].dt.normalize())["id_vaca"].nunique()
df_sistema["vacas_activas_dia"] = df_sistema["bloque_tiempo"].dt.normalize().map(vacas_por_dia)

df_sistema["ordenios_otros_robots"] = (
    df_sistema.groupby("bloque_tiempo")["total_ordenios"].transform("sum") - df_sistema["total_ordenios"]
)

display(df_sistema[["ms", "bloque_tiempo", "hora", "dia_semana", "temporada", "min_desde_alimento",
                    "hubo_lavado", "ordenios_con_lavado", "vacas_activas_dia", "ordenios_otros_robots"]].head())


## Atributos del bloque anterior (rezagos)

`pct_ocupacion_robot`, `duracion_total_min` y `total_ordenios` se calculan con los mismos ordeños que la variable objetivo, por lo que al momento de predecir todavía no se conocen. Se agregan sus valores del **bloque anterior** del mismo robot, que sí estarían disponibles. Las columnas originales se conservan; qué versión se usa se decide en el modelado.

`df_sistema` solo tiene bloques con ordeños, así que antes del `shift(1)` se completa la malla con todos los bloques de 30 min de cada robot (los bloques vacíos tienen 0 ordeños). Así el bloque anterior es siempre el de 30 min antes. Los rezagos se calculan antes de quitar días, para que no salten sobre los huecos que deja el filtro.

In [ ]:
malla = pd.MultiIndex.from_product(
    [sorted(df_sistema["ms"].unique()),
     pd.date_range(df_sistema["bloque_tiempo"].min(), df_sistema["bloque_tiempo"].max(), freq=VENTANA)],
    names=["ms", "bloque_tiempo"],
)
completo = df_sistema.set_index(["ms", "bloque_tiempo"]).reindex(malla)
cols_cero = ["total_ordenios", "duracion_total_min", "pct_ocupacion_robot"]
completo[cols_cero] = completo[cols_cero].fillna(0)

g = completo.groupby(level="ms")
completo["ocupacion_bloque_anterior"] = g["pct_ocupacion_robot"].shift(1)
completo["ordenios_bloque_anterior"] = g["total_ordenios"].shift(1)
# Si el bloque anterior no tuvo ordeños, no hubo cola
completo["pct_cola_bloque_anterior"] = g["pct_posible_cola"].shift(1).where(completo["ordenios_bloque_anterior"] > 0, 0)

# Regresar solo a los bloques con ordeños
df_sistema = completo[completo["total_ordenios"] > 0].reset_index()
# El reindex convierte los conteos a decimales: se regresan a enteros
cols_enteras = ["total_ordenios", "patadas_totales", "incompletos_totales", "pezones_no_enc_totales",
                "ordenios_con_lavado", "hubo_lavado", "hora", "dia_semana",
                "vacas_activas_dia", "ordenios_otros_robots", "ordenios_bloque_anterior"]
df_sistema[cols_enteras] = df_sistema[cols_enteras].astype("Int64")

print(f"Bloques en la malla completa: {len(completo):,} | bloques con ordeños: {len(df_sistema):,}")
display(df_sistema[["ms", "bloque_tiempo", "pct_ocupacion_robot", "ocupacion_bloque_anterior",
                    "total_ordenios", "ordenios_bloque_anterior",
                    "pct_posible_cola", "pct_cola_bloque_anterior"]].head(8))


## Subconjuntos finales

Se aplican al `df_sistema` definitivo los criterios de la sección de selección y de la limpieza, y se registra cuántos bloques quita cada uno:

1. Días con operación incompleta (`dia_completo`, calculado en la sección de selección).
2. Meses incompletos del historial (julio 2022 y enero 2026).
3. Bloques con menos de `MIN_ORDENIOS_BLOQUE` ordeños. Con el valor actual (1) no se quita ningún bloque, porque se decidió conservar los bloques de un ordeño.

In [ ]:
df_sistema_sel = df_sistema.copy()
df_sistema_sel["dia"] = df_sistema_sel["bloque_tiempo"].dt.normalize()
df_sistema_sel = df_sistema_sel.merge(diario[["ms", "dia", "dia_completo"]], on=["ms", "dia"], how="left")

registro = [("Bloques de df_sistema", len(df_sistema_sel))]

df_sistema_sel = df_sistema_sel[df_sistema_sel["dia_completo"].eq(True)]
registro.append(("Quitar días con operación incompleta", len(df_sistema_sel)))

mes_bloque = df_sistema_sel["bloque_tiempo"].dt.to_period("M").astype(str)
df_sistema_sel = df_sistema_sel[~mes_bloque.isin(MESES_EXCLUIR)]
registro.append((f"Quitar meses incompletos ({', '.join(MESES_EXCLUIR)})", len(df_sistema_sel)))

df_sistema_sel = df_sistema_sel[df_sistema_sel["total_ordenios"] >= MIN_ORDENIOS_BLOQUE]
registro.append((f"Quitar bloques con menos de {MIN_ORDENIOS_BLOQUE} ordeños", len(df_sistema_sel)))

df_sistema_sel = df_sistema_sel.drop(columns=["dia", "dia_completo"]).reset_index(drop=True)

tabla_seleccion = pd.DataFrame(registro, columns=["paso", "bloques"])
tabla_seleccion["quitados"] = (-tabla_seleccion["bloques"].diff()).fillna(0).astype(int)
tabla_seleccion["pct_restante"] = (tabla_seleccion["bloques"] / tabla_seleccion["bloques"].iloc[0] * 100).round(1)
display(tabla_seleccion)

print(f"df_sistema_sel: {len(df_sistema_sel):,} bloques x {df_sistema_sel.shape[1]} columnas, "
      f"{df_sistema_sel.memory_usage(deep=True).sum() / 1024**2:.1f} MB en memoria")
print(list(df_sistema_sel.columns))


# Formatear los datos

> Se ajusta el formato de `df_sistema_sel` para que scikit-learn lo pueda leer, esto sin cambiar el significado de los datos.

1. Tipos : Las columnas `Int64` de pandas fueron reemplazadas por `int64`
2. Valores Faltantes: 9,010 de los bloques no tienen etapa de lactancia conocida, se agregó la columna `sin_etapa`, con el propósito de marcar los bloques con etapas vacías.
3. Orden de los registros: Se ordenan por `bloque_tiempo` y `ms` debido a que `TimeSeriesSplit` y los valores de los bloques anteriores requieren orden cronológico.  
4. Orden de los atributos: Identificadores primero, variables de entrada en el medio y finalmente la variable objetivo.


In [ ]:
print(df_sistema_sel.dtypes)
print(df_sistema_sel.isna().sum())
print(df_sistema_sel["bloque_tiempo"].is_monotonic_increasing)

Como se observa en la muestra podemos observar que algunas columnas se encuentran en el formato de `Int64` el cual es un tipo de formato especial de pandas, lo cual puede chocar con scikit-learn, es por eso que se cambia a `int64`.



In [ ]:
cols_Int64 = df_sistema_sel.select_dtypes(include="Int64").columns
df_sistema_sel[cols_Int64] = df_sistema_sel[cols_Int64].astype("int64")

Dentro de nuestro análisis inicial se pudo observar que existían 5 columnas de la etapa de lactancia con varios valores nulos (9,010), es por eso que se creó la columna `sin_etapa` con el propósito de marcar con 1 los bloques con etapas desconocidas y posteriormente se rellenaron los valores nulos con ceros. Con el fin, de no perder filas y que el modelo tuviera la oportunidad de distinguir cuales eran los valores desconocidos.


In [ ]:
cols_etapa = ["pct_fresca", "pct_primer_parto", "pct_alta", "pct_baja", "pct_extendida"]
df_sistema_sel["sin_etapa"] = df_sistema_sel["pct_fresca"].isna().astype(int)
df_sistema_sel[cols_etapa] = df_sistema_sel[cols_etapa].fillna(0)

Como se observó en la revisión inicial se obtuvo un valor False en el orden por fecha, es por eso que se utilizó sort.values para ordenar los registros por `bloque_tiempo` y `ms`.


In [ ]:
df_sistema_sel = df_sistema_sel.sort_values(["bloque_tiempo", "ms"]).reset_index(drop=True)

Se ordenó `df_sistema_sel` iniciando con las columnas identificadas, posteriormente las variables de entradas y finalmente la variable objetivo. Con el propósito de una organización más útil, aumentando la claridad del propósito de cada columna.


In [ ]:
IDENTIFICADORES = ['ms', 'bloque_tiempo']
OBJETIVO = "pct_posible_cola"
ENTRADAS = [ c for c in df_sistema_sel.columns if c not in IDENTIFICADORES + [OBJETIVO]]

df_sistema_sel = df_sistema_sel[IDENTIFICADORES + ENTRADAS + [OBJETIVO]]

Como podemos observar, la salida muestra el cambio del tipo de variable de `Int64` a `int64`, para que tenga una mejor compatibilidad con `scikit-learn`, por otro lado, notamos que ya no se encuentran valores nulos en las 5 columnas, esto debido a que fueron rellenadas con ceros. Finalmente, notamos que `is_monotonic_increasing` regresa True, por lo cual nos da a saber que nuestros valores en los bloques de tiempo están ordenados. Cabe mencionar, que el orden de `df_sistema_sel` empieza con las columnas de identificadores, continúa con las variables de entrada y finalmente termina con la variable objetivo debido a que esto aumenta la claridad.


In [ ]:
print(df_sistema_sel.dtypes)
print(df_sistema_sel.isna().sum())
print(df_sistema_sel["bloque_tiempo"].is_monotonic_increasing)

Se guardaron los datos formateados en Datos / Procesado con el nombre de `df_sistema_formateado.parquet`

In [ ]:
df_sistema_sel.to_parquet(os.path.join(RUTA_DATOS, "df_sistema_fromateado.parquet"), index = False)
print(f"Guardado: {len(df_sistema_sel):,} bloques x {df_sistema_sel.shape[1]} columnas")

# Normalización de datos

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import pandas as pd

TARGET = "pct_posible_cola"

df_model = df_sistema_sel.copy()

columnas_excluir = [TARGET, "ms", "bloque_tiempo"]

X = df_model.drop(columns=columnas_excluir)
y = df_model[TARGET]

categoricas = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
numericas = X.select_dtypes(include=["number"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categoricas),
    ],
    remainder="drop",
    verbose_feature_names_out=False,   # quita los prefijos num__ / cat__
)

# Que devuelva un DataFrame directamente (sklearn >= 1.2)
preprocessor.set_output(transform="pandas")

X_normalizado = preprocessor.fit_transform(X)

In [ ]:
no_normalizadas = df_model[columnas_excluir]   # TARGET, ms, bloque_tiempo

# Reintegrar por índice
df_final = pd.concat([no_normalizadas, X_normalizado], axis=1)

# Validación

Se usa el mismo `TimeSeriesSplit` de la sección de muestreo, ahora sobre el `df_sistema_sel` definitivo: cada fold entrena con el pasado y prueba con el periodo siguiente.

In [ ]:
df_final = df_final.sort_values(["bloque_tiempo", "ms"]).reset_index(drop=True)

folds = []
for i, (idx_ent, idx_prueba) in enumerate(tscv.split(df_final), start=1):
    t = df_final["bloque_tiempo"]
    folds.append({
        "fold": i,
        "entrenamiento": f"{t.iloc[idx_ent[0]].date()} a {t.iloc[idx_ent[-1]].date()}",
        "prueba": f"{t.iloc[idx_prueba[0]].date()} a {t.iloc[idx_prueba[-1]].date()}",
        "bloques_entrenamiento": len(idx_ent),
        "bloques_prueba": len(idx_prueba),
    })
display(pd.DataFrame(folds))


# Dataset Final

El dataset fianl para la fase del modelado parte de `df_sistema_del` el cual esta formateado, no esta normalizado debido a que esta sera realizada en la etapa del modelo ajustandolo con los datos de entrenamiento de cada fold.

Se conservan todas las columnas excepto `incompletos_totales`, `pezones_no_enc_totales` y `duracion_promedio_min`, que se excluyeron en la base lógica de inclusión/exclusión por tener una correlación menor a 0.1 con la formación de colas.

In [ ]:
EXCLUIDAS = ["incompletos_totales", "pezones_no_enc_totales", "duracion_promedio_min"]

dataset_final = df_sistema_sel.drop(columns=EXCLUIDAS).copy()
print(f"Columnas antes: {df_sistema_sel.shape[1]} | después: {dataset_final.shape[1]}")
print(list(dataset_final.columns))